# 📖 Notebook 10: Production Patterns — Scaling, Multi-Tenancy, and Disaster Recovery

Welcome to the final notebook in the Kubernetes lab series. Up to this point, you have learned how to build, expose, secure, observe, and persist workloads. Now we step into the production mindset: how do you keep systems available, efficient, isolated, and recoverable when real traffic and real failures show up?

> **Prerequisites: Notebooks 01–03.** The HPA exercise scales the `api-gateway` Deployment
> in `k8s-lab` and drives load through its Service.

In [ ]:
# ── Preflight ────────────────────────────────────────────────────────────
# Every later cell shells out to these tools. Without this check a missing
# binary fails silently inside a `!` magic and you only see a confusing
# downstream error (e.g. FileNotFoundError from %%writefile) instead of
# "helm is not installed". Run this first.
import shutil
import subprocess

REQUIRED = ['minikube', 'kubectl', 'helm']
INSTALL_HINTS = {
    'minikube': 'https://minikube.sigs.k8s.io/docs/start/  (or `brew install minikube`)',
    'kubectl': 'https://kubernetes.io/docs/tasks/tools/  (or `brew install kubectl`)',
    'helm': 'https://helm.sh/docs/intro/install/  (or `brew install helm`)',
}

missing = [b for b in REQUIRED if shutil.which(b) is None]
if missing:
    hint = '\n'.join(f'  - {b}: {INSTALL_HINTS[b]}' for b in missing)
    raise RuntimeError(
        f"Missing required CLI tool(s): {', '.join(missing)}\n"
        f"Install them, then re-run this cell:\n{hint}"
    )

# A reachable cluster is required too -- `kubectl` alone is not enough.
probe = subprocess.run(
    ['kubectl', 'cluster-info'], capture_output=True, text=True
)
if probe.returncode != 0:
    raise RuntimeError(
        'No reachable Kubernetes cluster. Start the one from notebook 1:\n'
        '  minikube start --cpus=4 --memory=6144 --driver=docker\n'
        f'kubectl said: {probe.stderr.strip()[:300]}'
    )

print('Preflight OK:', ', '.join(REQUIRED), '+ cluster reachable')

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────
# `!kubectl ...` prints but never fails a cell. Everything this notebook claims
# -- QoS classes, OOMKills, CPU throttling, autoscaling -- is checked in Python
# as well, so a demo that stops reproducing its own lesson fails loudly.
import json
import subprocess
import time

NS = "k8s-lab"


def kget(*args, ns=NS):
    cmd = ["kubectl", "get", *args, "-o", "json"] + (["-n", ns] if ns else [])
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("kubectl failed: " + r.stderr.strip()[:400])
    return json.loads(r.stdout)


def wait_until(predicate, timeout=170, interval=5, what="condition"):
    deadline = time.time() + timeout
    while time.time() < deadline:
        value = predicate()
        if value:
            return value
        time.sleep(interval)
    raise AssertionError(f"timed out after {timeout}s waiting for {what}")


def hpa_line():
    """One row of `kubectl get hpa`, as text -- what a human would read."""
    return subprocess.run(
        ["kubectl", "get", "hpa", "api-gateway-hpa", "-n", NS, "--no-headers"],
        capture_output=True, text=True).stdout.strip()


def hpa_state():
    """(current cpu utilisation or None, current replicas, desired replicas)."""
    h = kget("hpa", "api-gateway-hpa")
    cpu = None
    for m in h["status"].get("currentMetrics") or []:
        if m.get("resource", {}).get("name") == "cpu":
            cpu = m["resource"]["current"].get("averageUtilization")
    return cpu, h["status"].get("currentReplicas"), h["status"].get("desiredReplicas")


print("helpers ready")

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain **resource requests** and **resource limits**
- Recognize Kubernetes **QoS classes**: Guaranteed, Burstable, and BestEffort
- Use a **HorizontalPodAutoscaler (HPA)** to scale based on usage
- Explain when **Vertical Pod Autoscaler (VPA)** is useful
- Create a **PodDisruptionBudget (PDB)** for critical workloads
- Describe common **multi-tenancy** patterns using namespaces, quotas, and policies
- Explain how **Velero** helps with backup and restore
- Build a simple production readiness checklist for Kubernetes services

## 🛠️ Setup

Before you begin:

- Start Minikube and make sure the `k8s-lab` namespace is still available
- Run this lab from `03-technologies/container-orchestration/kubernetes/` so the reference manifests are easy to find
- Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → 'Reload Window'.

This notebook assumes the sample FastAPI microservices are already running:

- `api-gateway` on port `8000`
- `user-service` on port `8001`
- `order-service` on port `8002`

We also need the Kubernetes metrics pipeline, because HPA depends on usage metrics such as CPU.

In [ ]:
!kubectl config current-context
!kubectl get ns k8s-lab
!kubectl get pods -n k8s-lab

# An HPA with `type: Resource` reads the metrics.k8s.io API, which is served by
# metrics-server and by nothing else. Without it the HPA reports
# `TARGETS: <unknown>/60%` and never scales -- installing Prometheus does not help.
!minikube addons enable metrics-server
!kubectl wait --for=condition=ready pod -l k8s-app=metrics-server -n kube-system --timeout=180s
!kubectl get apiservice v1beta1.metrics.k8s.io

# `kubectl top` returns "Metrics API not available" for ~60s after enabling the
# addon: metrics-server needs a couple of scrape intervals before it can report.
!kubectl top pods -n k8s-lab || echo 'Metrics may take a minute to appear after enabling metrics-server.'

## Production vs Development

A development cluster proves that your app can run. A production cluster must prove that your app can survive.

In development, you might be okay with:

- one replica
- weak defaults
- manual restarts
- no backup story

In production, you care about:

- predictable resource usage
- automatic scaling
- safe maintenance windows
- tenant isolation
- disaster recovery

This notebook is about those patterns.

## 📦 Resource Requests and Limits

Every container competes for CPU and memory on a node. Kubernetes needs hints so it can
schedule workloads safely.

- **Request** = what the **scheduler** reserves for the container. It is the only number
  used to decide which node the pod fits on. It is *not* a cap, and it is *not* measured
  against real usage.
- **Limit** = the ceiling the **kubelet and kernel** enforce at runtime. The scheduler
  never looks at it.

Think of requests as a reserved seat on a train, and limits as the maximum luggage
allowance.

### CPU and memory behave completely differently at the limit

| Resource | Exceeding the limit | Symptom |
|---|---|---|
| **CPU** — *compressible* | the container is **throttled** by the kernel's CFS scheduler | latency rises, nothing is killed, pod status looks perfectly healthy. The evidence is `nr_throttled` in the container's `cpu.stat`. |
| **Memory** — *incompressible* | the container is **OOMKilled** immediately | a restart, `Last State: Terminated / Reason: OOMKilled / Exit Code: 137`, then `CrashLoopBackOff` if it keeps happening. |

There is no such thing as throttled memory: you cannot ask a process to use a bit less RAM
for a while. That asymmetry is why the common production rule is **set memory limit ==
memory request, and be conservative with CPU limits** — a CPU limit throttles a healthy
service even when the node has idle cores sitting next to it.

### QoS class — derived from those numbers, never set by you

| QoS class | Exact condition | Eviction order under node pressure |
|---|---|---|
| **Guaranteed** | *every* container sets **both** request and limit for **both** cpu and memory, and `request == limit` for each | evicted **last**, lowest OOM score |
| **Burstable** | at least one request or limit is set, but the pod is not Guaranteed | in the middle — within this class, pods whose *usage exceeds their request by the most* go first |
| **BestEffort** | **no** container sets any request or limit | evicted **first**, always |

Two details people get wrong:

- The condition is per-**pod**, evaluated over **every container** (init containers
  included). One sidecar without limits drops the whole pod out of Guaranteed.
- Setting only `requests` gives you Burstable, not Guaranteed — and setting only `limits`
  makes Kubernetes copy them into requests, which *does* give you Guaranteed. Surprising,
  but that is the defaulting rule.

QoS is not just an eviction ranking: it also decides the kernel `oom_score_adj` written
for the container's processes, so under host-level memory pressure BestEffort processes
are the ones the kernel reaps first.

### ✅ Exercise
Create one pod for each QoS class and inspect how Kubernetes labels them.

In [ ]:
%%writefile qos-demo.yaml
apiVersion: v1
kind: Pod
metadata:
  name: guaranteed-demo
  namespace: k8s-lab
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "sleep 3600"]
      resources:
        requests:
          cpu: 100m
          memory: 128Mi
        limits:
          cpu: 100m
          memory: 128Mi
---
apiVersion: v1
kind: Pod
metadata:
  name: burstable-demo
  namespace: k8s-lab
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "sleep 3600"]
      resources:
        requests:
          cpu: 100m
          memory: 64Mi
        limits:
          cpu: 300m
          memory: 256Mi
---
apiVersion: v1
kind: Pod
metadata:
  name: besteffort-demo
  namespace: k8s-lab
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "sleep 3600"]

In [ ]:
!kubectl delete pod guaranteed-demo burstable-demo besteffort-demo -n k8s-lab --ignore-not-found
!kubectl apply -f qos-demo.yaml
!kubectl wait --for=condition=Ready pod/guaranteed-demo -n k8s-lab --timeout=120s
!kubectl wait --for=condition=Ready pod/burstable-demo -n k8s-lab --timeout=120s
!kubectl wait --for=condition=Ready pod/besteffort-demo -n k8s-lab --timeout=120s

print()
# Nothing in qos-demo.yaml mentions a QoS class -- the API server derived all three
# from the requests/limits alone.
!kubectl get pod guaranteed-demo burstable-demo besteffort-demo -n k8s-lab -o custom-columns=NAME:.metadata.name,QOS:.status.qosClass,REQ_CPU:.spec.containers[0].resources.requests.cpu,LIM_CPU:.spec.containers[0].resources.limits.cpu,REQ_MEM:.spec.containers[0].resources.requests.memory,LIM_MEM:.spec.containers[0].resources.limits.memory

expected = {"guaranteed-demo": "Guaranteed",
            "burstable-demo": "Burstable",
            "besteffort-demo": "BestEffort"}
got = {name: kget("pod", name)["status"]["qosClass"] for name in expected}
assert got == expected, f"the API server derived {got}, expected {expected}"
print(f"\n✅ three pods, three classes, derived purely from the numbers: {got}")

## 💥 Failure First: What "Over the Limit" Actually Looks Like

The table above is easy to nod along to. Let's watch both failures happen, because they
look nothing like each other in `kubectl`.

### 1. Memory over the limit → OOMKilled

A container with `limits.memory: 128Mi` that tries to allocate 300 MB. There is no grace
period and no warning: the kernel kills it the instant the cgroup limit is hit.

In [ ]:
%%writefile oom-demo.yaml
apiVersion: v1
kind: Pod
metadata:
  name: oom-demo
  namespace: k8s-lab
spec:
  restartPolicy: Always
  containers:
    - name: hog
      # Reuse the image notebook 02 built into minikube -- it has Python and is
      # already local, so nothing is pulled.
      image: k8s-lab/user-service:latest
      imagePullPolicy: IfNotPresent
      command: ["python", "-c", "import time; buf = bytearray(300 * 1024 * 1024); time.sleep(600)"]
      resources:
        requests:
          memory: 128Mi
        limits:
          memory: 128Mi

In [ ]:
!kubectl delete pod oom-demo -n k8s-lab --ignore-not-found
!kubectl apply -f oom-demo.yaml

# Poll rather than sleeping a fixed 25 seconds: the first kill can take a moment
# on a busy machine, and a fixed sleep either wastes time or misses the event.


def oom_status():
    pod = kget("pod", "oom-demo")
    for cs in pod["status"].get("containerStatuses", []):
        terminated = cs.get("lastState", {}).get("terminated") or {}
        if terminated.get("reason") == "OOMKilled":
            return cs
    return None


status = wait_until(oom_status, timeout=170, interval=3,
                    what="the container to be OOMKilled")

print("\n--- kubectl get pods ---")
!kubectl get pod oom-demo -n k8s-lab

print("\n--- the actual verdict ---")
!kubectl get pod oom-demo -n k8s-lab -o jsonpath='{range .status.containerStatuses[*]}restartCount={.restartCount}{"\n"}lastState={.lastState}{"\n"}{end}'
print()
!kubectl describe pod oom-demo -n k8s-lab | grep -A6 'Last State'

term = status["lastState"]["terminated"]
assert term["reason"] == "OOMKilled", term
assert term["exitCode"] == 137, f"expected 137 (128 + SIGKILL's 9), got {term['exitCode']}"
assert status["restartCount"] >= 1, "the container should have been restarted after the kill"

# And note what did NOT happen: the pod was never evicted. An OOMKill inside a
# container's own limit is a per-container event; the node was never under
# pressure, so nothing appears at the node or scheduler level.
pod = kget("pod", "oom-demo")
assert pod["status"]["phase"] == "Running", \
    f"the pod itself should still exist and be restarting, not {pod['status']['phase']}"
print(f"\n✅ OOMKilled, exit code {term['exitCode']}, "
      f"{status['restartCount']} restart(s) -- and no eviction")

!kubectl delete pod oom-demo -n k8s-lab --ignore-not-found
!rm -f oom-demo.yaml

`Reason: OOMKilled`, `Exit Code: 137` (128 + SIGKILL's signal number 9). The container is
restarted, hits the limit again, and after a few rounds the pod settles into
`CrashLoopBackOff` with an exponential back-off.

Note what did **not** happen: no eviction, no rescheduling, no event on the node. Only
*this container* was killed — the node was never under pressure. An OOMKill inside a limit
is a per-container event, which is why it is easy to miss in cluster-level dashboards and
why alerting on `kube_pod_container_status_last_terminated_reason{reason="OOMKilled"}` is
worth setting up.

### 2. CPU over the limit → throttled, and nothing else

Now the same idea with CPU: a busy loop capped at `limits.cpu: 100m`, which is one tenth
of a core. The kernel gives it 10 ms of CPU out of every 100 ms period and stops it for
the other 90 ms. The pod stays `Running` and `Ready` throughout — the only visible trace
is the throttling counter in the container's cgroup.

In [ ]:
%%writefile throttle-demo.yaml
apiVersion: v1
kind: Pod
metadata:
  name: throttle-demo
  namespace: k8s-lab
spec:
  containers:
    - name: spin
      image: busybox:1.36
      # A tight loop that would happily use a whole core if allowed to.
      command: ["sh", "-c", "while true; do :; done"]
      resources:
        requests:
          cpu: 100m
        limits:
          cpu: 100m

In [ ]:
!kubectl delete pod throttle-demo -n k8s-lab --ignore-not-found
!kubectl apply -f throttle-demo.yaml
!kubectl wait --for=condition=Ready pod/throttle-demo -n k8s-lab --timeout=120s

import time
time.sleep(30)

print("--- pod status: completely healthy ---")
!kubectl get pod throttle-demo -n k8s-lab

print("\n--- usage: pinned at the 100m ceiling ---")
!kubectl top pod throttle-demo -n k8s-lab || echo "(metrics-server still warming up)"

print("\n--- cpu.stat from inside the container (cgroup v2, then v1) ---")
print("nr_throttled = how many CFS periods the container was stopped in")
print("throttled_usec / throttled_time = total time spent stopped\n")
!kubectl exec -n k8s-lab throttle-demo -- sh -c "cat /sys/fs/cgroup/cpu.stat 2>/dev/null || cat /sys/fs/cgroup/cpu/cpu.stat 2>/dev/null || echo '(cgroup layout differs on this node)'"

# Both halves of the claim, checked. First: the pod looks perfectly healthy.
pod = kget("pod", "throttle-demo")
cs = pod["status"]["containerStatuses"][0]
assert pod["status"]["phase"] == "Running" and cs["ready"] and cs["restartCount"] == 0, \
    f"CPU pressure must not kill or restart anything: {cs}"

# Second: the kernel really is stopping it, and the only evidence is in cgroup
# counters that nothing in `kubectl` surfaces.
stat = subprocess.run(
    ["kubectl", "exec", "-n", NS, "throttle-demo", "--", "sh", "-c",
     "cat /sys/fs/cgroup/cpu.stat 2>/dev/null || cat /sys/fs/cgroup/cpu/cpu.stat"],
    capture_output=True, text=True).stdout
values = dict(line.split()[:2] for line in stat.strip().splitlines() if len(line.split()) >= 2)
throttled = int(values.get("nr_throttled", 0))
assert throttled > 0, (
    "a busy loop capped at 100m should be throttled in most CFS periods; "
    f"nr_throttled is {throttled}. cpu.stat was:\n{stat}"
)
print(f"\n✅ Running, Ready, 0 restarts -- and throttled in {throttled} CFS periods")

!kubectl delete pod throttle-demo -n k8s-lab --ignore-not-found
!rm -f throttle-demo.yaml

A large, steadily climbing `nr_throttled` on a pod that looks perfectly healthy is the
signature of a CPU limit set too low. Users see it as latency; `kubectl get pods` shows
nothing at all. This is the single most common cause of "the service is slow but nothing
is wrong".

It is also why many teams set CPU **requests** and deliberately omit CPU **limits** on
latency-sensitive services: the request guarantees a floor, and without a limit the
container can use idle CPU that would otherwise go to waste. The trade-off is that a
runaway process can then starve its neighbours, so the technique needs requests to be set
correctly on *everything* on the node.

## 🔁 HorizontalPodAutoscaler (HPA)

An **HPA** means: **more pods when busy, fewer pods when idle**.

Instead of making one pod bigger, HPA adds or removes replicas. This usually works very well for stateless services like our `api-gateway`.

### Where the numbers come from

`../manifests/hpa.yaml` uses `apiVersion: autoscaling/v2` — the version to use. (`v1`
supported only a single CPU target and is long superseded; `v2beta1` was removed in
Kubernetes 1.25 and `v2beta2` in 1.26.) `v2` gives you three metric sources:

| `type` | Read from | Needs |
|---|---|---|
| `Resource` | `metrics.k8s.io` — cpu/memory of the target's own pods | **metrics-server** |
| `Pods` / `Object` | `custom.metrics.k8s.io` — e.g. requests per second | a custom metrics adapter (`prometheus-adapter`) |
| `External` | `external.metrics.k8s.io` — e.g. queue depth in SQS | an external metrics adapter |

Our HPA uses `type: Resource`, so **metrics-server is mandatory** and Prometheus is
irrelevant to it. A missing metrics-server shows up as `TARGETS: <unknown>/60%` and an
HPA that never acts.

### The formula, and why requests matter here too

```text
desiredReplicas = ceil( currentReplicas x ( currentMetricValue / desiredMetricValue ) )
```

For `averageUtilization`, `currentMetricValue` is **usage as a percentage of the pod's
CPU *request***, not of a core and not of the limit. A pod with no CPU request has no
denominator, so the HPA cannot compute a utilisation at all and reports `<unknown>`
forever. **An HPA on CPU utilisation is only as good as the requests on the pods it
scales.**

Our manifest also sets `behavior`, which is what stops autoscaler flapping:
`scaleUp.stabilizationWindowSeconds: 30` (react quickly), `scaleDown` 300 (retreat
slowly). Scaling down fast is how you end up oscillating.

Finally: with two metrics listed (cpu and memory), the HPA computes a desired replica
count for **each** and takes the **highest**. Memory rarely falls after load stops, so a
memory metric can quietly pin your replica count at its high-water mark.

### ✅ Exercise
Apply the HPA, generate traffic, and observe the replica count change.

In [ ]:
# This notebook runs from `notebooks/`, so the shared manifests are one level up.
!kubectl apply -f ../manifests/hpa.yaml
!kubectl get hpa api-gateway-hpa -n k8s-lab
!kubectl get deployment api-gateway -n k8s-lab

# An HPA needs a moment before it can report anything, and if it never can, the
# whole exercise below is a five-minute wait for nothing. `TARGETS: <unknown>/60%`
# forever means metrics-server is missing or the target pods have no CPU request.
cpu = wait_until(lambda: hpa_state()[0] is not None, timeout=170, interval=5,
                 what="the HPA to compute a CPU utilisation")
print(f"\nHPA is reading metrics: current CPU utilisation {hpa_state()[0]}% "
      "against the 60% target")

target = kget("deployment", "api-gateway")["spec"]["template"]["spec"]["containers"][0]
assert "cpu" in target["resources"]["requests"], (
    "api-gateway has no CPU request, so `averageUtilization` has no denominator "
    "and this HPA could never compute a value"
)
print(f"✅ HPA active; utilisation is a percentage of the pod's "
      f"{target['resources']['requests']['cpu']} CPU request")

In [ ]:
# Generate enough load to push api-gateway past 60% of its 100m CPU request.
# One busybox wget loop is not reliably enough: every iteration forks a process,
# so the client becomes the bottleneck long before the server does. Two
# generators hitting `/api/users` -- the route where the gateway does real work,
# an outbound HTTP call to user-service -- is comfortably over the line.
#
# Pinned tag (an unpinned `busybox` means "whatever :latest is today").
LOAD_GENS = ("load-gen", "load-gen-2")
LOAD_CMD = ("while true; do wget -q -O- "
            "http://api-gateway.k8s-lab:8000/api/users >/dev/null 2>&1; done")

for name in LOAD_GENS:
    subprocess.run(["kubectl", "delete", "pod", name, "-n", NS,
                    "--ignore-not-found", "--wait=true"], check=False)
    r = subprocess.run(["kubectl", "run", name, "-n", NS, "--image=busybox:1.36",
                        "--restart=Never", "--", "/bin/sh", "-c", LOAD_CMD],
                       capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip()[:200])

for name in LOAD_GENS:
    subprocess.run(["kubectl", "wait", "--for=condition=Ready", f"pod/{name}",
                    "-n", NS, "--timeout=120s"], check=False)

running = [p for p in kget("pods")["items"]
           if p["metadata"]["name"] in LOAD_GENS and p["status"]["phase"] == "Running"]
assert len(running) == len(LOAD_GENS), \
    f"only {len(running)} of {len(LOAD_GENS)} load generators started"
print(f"\n✅ {len(running)} load generators hammering api-gateway")

In [ ]:
# `kubectl get -w` inside a `!` magic never returns -- the cell hangs until you
# interrupt the kernel. Poll for a bounded time instead, so the notebook can be
# run top to bottom without manual intervention. The HPA controller re-evaluates
# every 15 seconds and `scaleUp.stabilizationWindowSeconds` is 30, so give it a
# couple of minutes.
observations = []

print(f"{'time':>5}  hpa targets / replicas")
for t in range(0, 150, 15):
    print(f"{t:>4}s  {hpa_line()}")
    observations.append(hpa_state())
    time.sleep(15)

print("\nHPA events (this is where 'why did/didn't it scale' is answered):")
!kubectl describe hpa api-gateway-hpa -n k8s-lab | tail -n 15

In [ ]:
# Keep watching until the replica count actually moves, then assert it did.
# "Watch the TARGETS column climb and REPLICAS follow it" is the claim this whole
# section rests on; without a check, an HPA that never scaled would look exactly
# like one that did and the reader would learn the wrong thing.
MIN_REPLICAS = kget("hpa", "api-gateway-hpa")["spec"]["minReplicas"]


def scaled_up():
    state = hpa_state()
    observations.append(state)
    print(f"  cpu={state[0]}%  replicas={state[1]}  desired={state[2]}")
    return state if (state[2] or 0) > MIN_REPLICAS else None


try:
    cpu, current, desired = wait_until(scaled_up, timeout=170, interval=15,
                                       what="the HPA to ask for more replicas")
finally:
    peak_cpu = max((c for c, _, _ in observations if c is not None), default=None)
    print(f"\npeak observed CPU utilisation: {peak_cpu}% (target 60%)")

assert peak_cpu is not None and peak_cpu > 60, (
    f"the load never pushed utilisation past the 60% target (peak {peak_cpu}%), so "
    "the HPA had no reason to scale. More load generators, or a smaller CPU request."
)
assert desired > MIN_REPLICAS, \
    f"utilisation exceeded the target but the HPA still wants {desired} replicas"

# And the Deployment follows the HPA, not the other way round.
dep = wait_until(
    lambda: (lambda d: d if d["spec"]["replicas"] > MIN_REPLICAS else None)(
        kget("deployment", "api-gateway")),
    timeout=120, interval=5, what="the Deployment's replica count to follow the HPA")
!kubectl get hpa api-gateway-hpa -n k8s-lab
!kubectl get deployment api-gateway -n k8s-lab
print(f"\n✅ load drove CPU to {peak_cpu}% of the request and the HPA scaled "
      f"{MIN_REPLICAS} -> {dep['spec']['replicas']} replicas")

The first loop above samples for two and a half minutes; the cell after it keeps waiting
until the replica count actually moves, and **fails the notebook if it never does**. Watch
the `TARGETS` column climb past the 60% CPU target and `REPLICAS` follow it upward.

If `TARGETS` shows `<unknown>/60%` the whole time, the cause is one of exactly two things:
metrics-server is not ready yet, or the target pods have no CPU **request** for the
percentage to be computed against.

Now stop the synthetic traffic and watch scale-down — which is deliberately much slower,
because `scaleDown.stabilizationWindowSeconds: 300` makes the HPA wait five minutes of
sustained low usage before it removes a pod.

In [ ]:
for name in LOAD_GENS:
    subprocess.run(["kubectl", "delete", "pod", name, "-n", NS,
                    "--ignore-not-found", "--wait=false"], check=False)

!kubectl get hpa api-gateway-hpa -n k8s-lab
!kubectl get deployment api-gateway -n k8s-lab

print()
print("Scale-down is intentionally slow: with stabilizationWindowSeconds=300 the HPA")
print("waits 5 minutes of low usage before removing a pod, and the scaleDown policy")
print("then removes at most 1 pod per minute. Re-run this cell in ~6 minutes to see it.")

# Which is a claim about the manifest, so read it back rather than trusting the
# prose: this is the parameter that stops an autoscaler oscillating.
behavior = kget("hpa", "api-gateway-hpa")["spec"]["behavior"]
assert behavior["scaleDown"]["stabilizationWindowSeconds"] == 300, behavior["scaleDown"]
assert behavior["scaleUp"]["stabilizationWindowSeconds"] == 30, behavior["scaleUp"]
print(f"\nscaleUp window {behavior['scaleUp']['stabilizationWindowSeconds']}s, "
      f"scaleDown window {behavior['scaleDown']['stabilizationWindowSeconds']}s "
      "-- react fast, retreat slowly")

## 📈 Vertical Pod Autoscaler (VPA) — Concept Only

If HPA means **more pods**, VPA means **bigger or smaller pods**. A VPA watches usage and recommends or sets new CPU and memory requests.

Use cases:

- **HPA** is great when the workload scales horizontally
- **VPA** is useful when a workload cannot easily be split into more replicas or when you want better sizing recommendations

A simple rule of thumb:

- web APIs: usually HPA first
- singleton workloads, batch jobs, or memory-heavy services: VPA may help

### ✅ Exercise
Look at `api-gateway`, `user-service`, and `order-service`. Which one would you scale horizontally first? Which one might benefit from better vertical sizing? Write down your answer before moving on.

## 🛡️ PodDisruptionBudget

A **PodDisruptionBudget (PDB)** tells Kubernetes: *during voluntary disruptions, always
keep at least this many pods available*.

The word doing the work is **voluntary**. A PDB is enforced by the **Eviction API**, and
only things that go through the Eviction API respect it:

| Respects a PDB (voluntary) | Ignores a PDB (involuntary) |
|---|---|
| `kubectl drain` during node maintenance | a node losing power or its kubelet dying |
| cluster-autoscaler removing an underused node | an OOMKill |
| a node-pool upgrade rolling nodes | the scheduler preempting a lower-priority pod |
| anything calling `POST /eviction` | you running `kubectl delete pod` |

That last row surprises people: `kubectl delete pod` is a plain DELETE, not an eviction,
so it bypasses the PDB entirely. A PDB is not a "do not delete this" guard.

### `minAvailable` vs `maxUnavailable`

Specify exactly one. Both accept a number or a percentage of `.spec.replicas`.

- `minAvailable: 1` — at most `replicas - 1` pods may be evicted at once.
- `maxUnavailable: 1` — the same thing when replicas is 2, and *not* the same thing when
  the replica count changes. Prefer `maxUnavailable` for services that autoscale, since it
  stays meaningful as `replicas` moves.

### The way PDBs actually cause outages

**`minAvailable` equal to the replica count deadlocks the drain.** With `replicas: 1` and
`minAvailable: 1`, no pod may ever be evicted, so `kubectl drain` blocks forever and a
node upgrade stalls cluster-wide. The same happens with `minAvailable: 2` on a
2-replica Deployment. A PDB is only meaningful when it leaves at least one pod evictable —
which means a service you want protected needs **at least 2 replicas** and a spread across
nodes.

And a PDB only counts pods that are **Ready**. If a rollout has left you with one Ready
pod out of two, `minAvailable: 1` already blocks eviction — correctly, but it means a
broken deployment can also stall node maintenance.

### ✅ Exercise
Create a PDB for `api-gateway` with `minAvailable: 1`, inspect it, and read the drain
command you would use during maintenance.

In [ ]:
%%writefile api-gateway-pdb.yaml
apiVersion: policy/v1
kind: PodDisruptionBudget
metadata:
  name: api-gateway-pdb
  namespace: k8s-lab
spec:
  minAvailable: 1
  selector:
    matchLabels:
      app: api-gateway

In [ ]:
!kubectl apply -f api-gateway-pdb.yaml
!kubectl get pdb -n k8s-lab

print()
# ALLOWED DISRUPTIONS is the number that matters: how many pods may be evicted
# right now. If it is 0, a drain of the node holding them will block.
!kubectl describe pdb api-gateway-pdb -n k8s-lab | grep -E 'Min available|Allowed disruptions|Current|Desired|Expected'
!kubectl get nodes

pdb = wait_until(
    lambda: (lambda p: p if p["status"].get("currentHealthy") else None)(
        kget("pdb", "api-gateway-pdb")),
    timeout=120, interval=3, what="the PDB controller to evaluate the budget")
status = pdb["status"]
print(f"\ncurrentHealthy={status['currentHealthy']}  "
      f"desiredHealthy={status['desiredHealthy']}  "
      f"disruptionsAllowed={status['disruptionsAllowed']}")

# The section's warning is that `minAvailable` equal to the replica count
# deadlocks a drain. Check we did NOT do that here: with more than one healthy
# pod and minAvailable: 1, at least one pod must remain evictable, or node
# maintenance would block forever on this Deployment.
assert status["currentHealthy"] >= 2, \
    "a PDB is only meaningful on a Deployment with at least 2 replicas"
assert status["disruptionsAllowed"] >= 1, (
    "disruptionsAllowed is 0 -- `kubectl drain` on this node would block forever. "
    "That is exactly the outage mode described above."
)
print(f"✅ {status['disruptionsAllowed']} pod(s) may be evicted at a time: "
      "maintenance can proceed without dropping below minAvailable")

In [ ]:
# Printed, not executed: draining the single node of a minikube cluster evicts
# everything you have built across ten notebooks.
!echo 'Optional maintenance test (single-node cluster -- this WILL evict your workloads):'
!echo '  kubectl drain minikube --ignore-daemonsets --delete-emptydir-data'
!echo '  kubectl uncordon minikube'
!echo ''
!echo 'Note the absence of --force. --force deletes pods that are not managed by a'
!echo 'controller, and it does NOT bypass PodDisruptionBudgets -- people often think'
!echo 'it does. To ignore a PDB you would have to delete the pods directly, which is'
!echo 'exactly the thing the PDB exists to stop you doing by accident.'

## 🏢 Multi-Tenancy

**Multi-tenancy** means multiple teams or applications share the same cluster without stepping on each other.

A common beginner-friendly pattern is **namespace-per-team**. Each team gets:

- its own namespace
- its own ResourceQuota
- its own LimitRange defaults
- its own NetworkPolicy rules

This is not perfect isolation like separate clusters, but it is a very common and practical starting point.

### ✅ Exercise
Create `team-a` and `team-b` namespaces with quotas, default limits, and a default-deny NetworkPolicy.

In [ ]:
%%writefile team-tenancy.yaml
apiVersion: v1
kind: Namespace
metadata:
  name: team-a
---
apiVersion: v1
kind: Namespace
metadata:
  name: team-b
---
apiVersion: v1
kind: ResourceQuota
metadata:
  name: team-a-quota
  namespace: team-a
spec:
  hard:
    pods: "5"
    requests.cpu: "1"
    requests.memory: 1Gi
    limits.cpu: "2"
    limits.memory: 2Gi
---
apiVersion: v1
kind: ResourceQuota
metadata:
  name: team-b-quota
  namespace: team-b
spec:
  hard:
    pods: "5"
    requests.cpu: "1"
    requests.memory: 1Gi
    limits.cpu: "2"
    limits.memory: 2Gi
---
apiVersion: v1
kind: LimitRange
metadata:
  name: team-a-defaults
  namespace: team-a
spec:
  limits:
    - type: Container
      default:
        cpu: 500m
        memory: 256Mi
      defaultRequest:
        cpu: 100m
        memory: 128Mi
---
apiVersion: v1
kind: LimitRange
metadata:
  name: team-b-defaults
  namespace: team-b
spec:
  limits:
    - type: Container
      default:
        cpu: 500m
        memory: 256Mi
      defaultRequest:
        cpu: 100m
        memory: 128Mi
---
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: default-deny
  namespace: team-a
spec:
  podSelector: {}
  policyTypes:
    - Ingress
    - Egress
---
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: default-deny
  namespace: team-b
spec:
  podSelector: {}
  policyTypes:
    - Ingress
    - Egress

In [ ]:
!kubectl apply -f team-tenancy.yaml
!kubectl get resourcequota -A
!kubectl get limitrange -A
!kubectl get networkpolicy -A

# A ResourceQuota is only real once the quota controller has computed its `used`
# figures; until then `kubectl describe` shows an empty budget and it is easy to
# believe a quota is in force when nothing has been evaluated yet.
for team in ("team-a", "team-b"):
    quota = wait_until(
        lambda t=team: (lambda q: q if q["status"].get("hard") else None)(
            kget("resourcequota", f"{t}-quota", ns=t)),
        timeout=120, interval=3, what=f"the {team} quota to be evaluated")
    assert quota["status"]["hard"]["pods"] == "5", quota["status"]["hard"]
    assert kget("limitrange", f"{team}-defaults", ns=team)["spec"]["limits"], \
        f"{team} has no LimitRange defaults"
    assert kget("networkpolicy", "default-deny", ns=team)["spec"]["policyTypes"] == \
        ["Ingress", "Egress"], "the default-deny policy should cover both directions"
print("\n✅ both tenants have an evaluated quota, default limits and a default-deny policy")

## 💽 Velero Backup and Restore

Backups are a production feature, not an afterthought. A Kubernetes backup strategy usually needs two things:

1. cluster resources such as Deployments, Services, and ConfigMaps
2. persistent data from volumes

**Velero** is a popular tool for this. It can back up Kubernetes objects and, depending on your storage setup, snapshot or copy persistent volume data too.

Velero needs an object-storage destination and credentials, which this lab does not set
up, so the commands below are **printed rather than executed**. They are the real
workflow — copy them when you have a bucket.

One distinction worth having straight before you rely on it: Velero backs up **API
objects** by default. Getting the *data* inside your PersistentVolumes requires either
CSI volume snapshots (fast, storage-backend-specific, usually same-region) or Velero's
file-system backup via the node-agent (slower, portable across providers). A backup
that captured your PVC objects but not their contents restores an empty database.

### ✅ Exercise
Add the Helm repo, review the chart, and read the backup, restore, and schedule commands below.

In [ ]:
# --force-update keeps this idempotent across re-runs.
!helm repo add vmware-tanzu https://vmware-tanzu.github.io/helm-charts --force-update
!helm repo update
!helm search repo vmware-tanzu/velero

In [ ]:
# These are PRINTED, not run. Velero is not installed by this lab (it needs an
# object-storage bucket and credentials -- S3, GCS, Azure Blob or MinIO -- plus
# the `velero` CLI), so executing them would just fail with "command not found".
!echo '# 1. Install the server side, pointed at a bucket:'
!echo 'velero install --provider aws --bucket my-velero-bucket \'
!echo '    --secret-file ./credentials-velero --backup-location-config region=eu-west-1 \'
!echo '    --plugins velero/velero-plugin-for-aws:v1.10.0'
!echo ''
!echo '# 2. Back up one namespace (Kubernetes objects; PV data needs snapshots or'
!echo '#    the node-agent / file-system backup enabled):'
!echo 'velero backup create k8s-lab-backup --include-namespaces k8s-lab --wait'
!echo 'velero backup describe k8s-lab-backup --details'
!echo ''
!echo '# 3. Restore:'
!echo 'velero restore create --from-backup k8s-lab-backup --wait'
!echo ''
!echo '# 4. Schedule it, because an unscheduled backup is a good intention:'
!echo 'velero schedule create daily-k8s-lab --schedule="0 2 * * *" \'
!echo '    --include-namespaces k8s-lab --ttl 720h'
!echo ''
!echo '# The only backup that counts is one you have restored. Test restores on a'
!echo '# schedule, into a scratch namespace, or you do not have a backup strategy --'
!echo '# you have a backup hope.'

## ✅ Production Checklist

Before calling a service production-ready, walk through this checklist:

- Resource **requests** on every container (the scheduler needs them; the HPA needs them)
- Memory **limit** equal to the memory request; CPU limits used sparingly and knowingly
- **Readiness** probe on everything that serves traffic — it is what gates Service
  endpoints and what makes `maxUnavailable: 0` mean anything (Notebook 02)
- **Liveness** probe that checks only this process, never a downstream dependency
- **Startup** probe instead of a large `initialDelaySeconds` for slow boots
- At least **2 replicas** and a PodDisruptionBudget that leaves one pod evictable
- **HPA** for variable-load services, with `behavior` tuned so it does not flap
- **NetworkPolicy** with a default-deny posture — and egress to UDP+TCP 53 allowed
- **RBAC** with least privilege; ClusterRole + RoleBinding rather than ClusterRoleBinding
- Secrets from an external store; Kubernetes Secrets are not encrypted at rest by default
- **GitOps** for deployment and drift correction
- Monitoring and alerting on signals that survive the failure (not `up == 0`)
- A backup strategy whose **restore** has actually been tested

If a team cannot explain how it handles each item above, it is not really production-ready yet.

## 🧹 Clean Up

The cell below removes only what **this** notebook created, so you can re-run the notebook
from the top without anything failing on `AlreadyExists`.

In [ ]:
# Remove just this notebook's objects. Everything uses --ignore-not-found, so
# this cell is safe to run twice and the notebook is safe to re-run from the top.
!kubectl delete pod guaranteed-demo burstable-demo besteffort-demo load-gen load-gen-2 oom-demo throttle-demo -n k8s-lab --ignore-not-found
!kubectl delete hpa api-gateway-hpa -n k8s-lab --ignore-not-found
!kubectl delete pdb api-gateway-pdb -n k8s-lab --ignore-not-found
!kubectl delete -f team-tenancy.yaml --ignore-not-found
!rm -f qos-demo.yaml api-gateway-pdb.yaml team-tenancy.yaml oom-demo.yaml throttle-demo.yaml

print()
!kubectl get pods -n k8s-lab

# The HPA scaled api-gateway up; removing the HPA does not scale it back, so put
# the replica count back where notebooks 02-09 expect it. (Deleting an HPA leaves
# the Deployment exactly as it last set it -- a small surprise that leaves clusters
# permanently over-provisioned after an autoscaling experiment.)
!kubectl scale deployment api-gateway -n k8s-lab --replicas=2

leftovers = [p["metadata"]["name"] for p in kget("pods")["items"]
             if p["metadata"]["name"].startswith(("guaranteed-", "burstable-",
                                                  "besteffort-", "load-gen",
                                                  "oom-demo", "throttle-demo"))]
assert not leftovers, f"this notebook's demo pods are still around: {leftovers}"
print("\n✅ demo objects removed, api-gateway back to 2 replicas")

### Tearing the whole thing down

When you are completely finished with the series, this deletes the cluster and everything
in it — all ten notebooks' worth of state. It is left commented out on purpose, because a
stray "Run All" should not destroy your lab.

```bash
minikube stop      # keeps everything; `minikube start` brings it all back
minikube delete    # destroys the cluster and all its data
```

In [ ]:
# Deliberately commented out -- uncomment only when you are done with the series.
# !minikube delete
print("Cluster left running. `minikube stop` to pause it, `minikube delete` to destroy it.")

## 🎓 What You Learned

You made it to the end of the 10-notebook Kubernetes lab series. Here is the big picture:

1. You created and explored a cluster
2. You deployed pods and deployments
3. You exposed services and networking paths
4. You packaged apps with Helm and Kustomize
5. You added observability
6. You locked things down with RBAC and network policies
7. You used GitOps ideas with ArgoCD
8. You explored service mesh concepts
9. You added storage and secrets
10. You learned production patterns for scaling, isolation, and recovery

That is a strong beginner-to-intermediate foundation. You now understand not just how to run containers in Kubernetes, but how to think like a platform engineer.

## 🚀 Where to Go Next

If you want to keep going, here are great next topics:

- Kubernetes official docs: workloads, networking, storage, and security
- Learn a GitOps workflow deeply with ArgoCD or Flux
- Practice production observability with Prometheus, Grafana, and Alertmanager
- Explore cluster security tools such as Kyverno, Falco, and image scanning
- Study platform engineering topics such as internal developer platforms and golden paths
- Try the same lab ideas on a managed cloud cluster such as AKS, EKS, or GKE

Most importantly: keep practicing. Kubernetes becomes much easier once you have seen the same ideas from the app side, the platform side, and the operations side.